In [1]:
import os, requests
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
API_URL = "https://api.weatherapi.com/v1/forecast.json"

import pandas as pd

In [2]:
def get_weather(city: str, days: int, url=API_URL, key=API_KEY):
    data = {}
    parameters = {'key':key, 'q':city, 'days':days}
    response = requests.get(url, params=parameters)
    
    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()
    else:
        print(f"Error fetching {city}.\n Response: {response.status_code}")
    return data

In [3]:
weather_data = get_weather('Varadero', 3)

In [4]:
weather_data['forecast']

{'forecastday': [{'date': '2026-05-01',
   'date_epoch': 1777593600,
   'day': {'maxtemp_c': 33.7,
    'maxtemp_f': 92.7,
    'mintemp_c': 22.7,
    'mintemp_f': 72.9,
    'avgtemp_c': 27.4,
    'avgtemp_f': 81.4,
    'maxwind_mph': 16.6,
    'maxwind_kph': 26.6,
    'totalprecip_mm': 0.0,
    'totalprecip_in': 0.0,
    'totalsnow_cm': 0.0,
    'avgvis_km': 10.0,
    'avgvis_miles': 6.0,
    'avghumidity': 76,
    'daily_will_it_rain': 0,
    'daily_chance_of_rain': 0,
    'daily_will_it_snow': 0,
    'daily_chance_of_snow': 0,
    'condition': {'text': 'Sunny',
     'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png',
     'code': 1000},
    'uv': 11.8},
   'astro': {'sunrise': '06:52 AM',
    'sunset': '07:53 PM',
    'moonrise': '08:07 PM',
    'moonset': '06:31 AM',
    'moon_phase': 'Full Moon',
    'moon_illumination': 99,
    'is_moon_up': 1,
    'is_sun_up': 0},
   'hour': [{'time_epoch': 1777608000,
     'time': '2026-05-01 00:00',
     'temp_c': 25.0,
     'temp_f': 76.9

In [5]:
forecast = weather_data['forecast']
forecast_3_days_summary = []

for day in forecast['forecastday']:
    forecast_3_days_summary.append(f"{day['date']}: High {day['day']['maxtemp_f']}F, Low {day['day']['mintemp_f']}F - {day['day']['condition']['text']}")

In [6]:
forecast_3_days_summary

['2026-05-01: High 92.7F, Low 72.9F - Sunny',
 '2026-05-02: High 90.2F, Low 72.9F - Sunny',
 '2026-05-03: High 87.8F, Low 77.1F - Patchy rain nearby']

In [7]:
forecast_3_days_summary = []

for day in forecast['forecastday']:
    format_day = datetime.strptime(day['date'], '%Y-%m-%d')
    formated_date = format_day.strftime('%A, %B %#d')
    forecast_3_days_summary.append(f"{formated_date}: High {day['day']['maxtemp_f']}F / Low {day['day']['mintemp_f']}F - {day['day']['condition']['text']}")

In [8]:
forecast_3_days_summary

['Friday, May 1: High 92.7F / Low 72.9F - Sunny',
 'Saturday, May 2: High 90.2F / Low 72.9F - Sunny',
 'Sunday, May 3: High 87.8F / Low 77.1F - Patchy rain nearby']

In [9]:
API_URL = "https://api.weatherapi.com/v1/astronomy.json"

<h2>One function to build summary of Weather Report</h2>

In [10]:
def get_data(url: str, parameters: dict):
    data = None
    response = requests.get(url, params=parameters)
    
    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()
    else:
        print(f"Error fetching data.\n Response: {response.status_code}")
    return data

In [11]:
def get_city_data(city: str, key=API_KEY):
    city_summary = {}
    parameters = {'key':key, 'q':city, 'days': 3}
    current_url = f"https://api.weatherapi.com/v1/current.json"
    forecast_url = f"https://api.weatherapi.com/v1/forecast.json"
    astronomy_url = f"https://api.weatherapi.com/v1/astronomy.json"
    
    current = get_data(current_url, parameters)
    forecast = get_data(forecast_url, parameters)
    astronomy = get_data(astronomy_url, parameters)
    
    city_name = current['location']['name']
    current_values = current['current']
    
    forecast_3_days_summary = []
    forecast_values = forecast['forecast']['forecastday']
    for day in forecast_values:
        format_day = datetime.strptime(day['date'], '%Y-%m-%d')
        formated_date = format_day.strftime('%A, %B %#d')
        forecast_3_days_summary.append(f"{formated_date}: High {day['day']['maxtemp_f']}F / Low {day['day']['mintemp_f']}F - {day['day']['condition']['text']}")
        
    astronomy_values = astronomy['astronomy']['astro']

    city_summary = {
        'city': city_name,
        'temp_f': current_values['temp_f'],
        'conditions': current_values['condition']['text'],
        'humidity': current_values['humidity'],
        'forecast': forecast_3_days_summary,
        'sunrise': astronomy_values['sunrise'],
        'sunset': astronomy_values['sunset'],
        'moon_phase': astronomy_values['moon_phase']
        }
    
    return city_summary

In [12]:
cities = [
    "New York City", "Los Angeles", "Chicago", "Houston", "Miami", 
    "Phoenix", "San Francisco", "Seattle", "Austin", "Denver",
    "Ciudad de Mexico", "Buenos Aires", "São Paulo", "Bogotá", "Lima",
    "Santiago", "Medellín", "Guadalajara", "Monterrey", "Quito"
]

In [13]:
summary = []
for city in cities:
    summary.append(get_city_data(city))

In [14]:
df = pd.DataFrame(summary)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   city        20 non-null     object 
 1   temp_f      20 non-null     float64
 2   conditions  20 non-null     object 
 3   humidity    20 non-null     int64  
 4   forecast    20 non-null     object 
 5   sunrise     20 non-null     object 
 6   sunset      20 non-null     object 
 7   moon_phase  20 non-null     object 
dtypes: float64(1), int64(1), object(6)
memory usage: 1.4+ KB


In [16]:
df[['city','temp_f']]

,city,temp_f
0,New York,55.9
1,Los Angeles,69.1
2,Chicago,45.0
3,Houston,59.7
4,Miami,82.9
5,Phoenix,89.1
6,San Francisco,62.1
7,Seattle,71.1
8,Austin,57.9
9,Denver,60.1


In [17]:
df

,city,temp_f,conditions,humidity,forecast,sunrise,sunset,moon_phase
0,New York,55.9,Sunny,57,"[Friday, May 1: High 70.7F / Low 42.1F - Sunny...",05:55 AM,07:52 PM,Full Moon
1,Los Angeles,69.1,Sunny,61,"[Friday, May 1: High 78.1F / Low 54.9F - Partl...",06:04 AM,07:37 PM,Full Moon
2,Chicago,45.0,Partly cloudy,58,"[Friday, May 1: High 49.6F / Low 38.7F - Patch...",05:47 AM,07:50 PM,Full Moon
3,Houston,59.7,Mist,100,"[Friday, May 1: High 64.6F / Low 58.1F - Heavy...",06:39 AM,07:59 PM,Full Moon
4,Miami,82.9,Sunny,63,"[Friday, May 1: High 89.4F / Low 68.4F - Sunny...",06:44 AM,07:52 PM,Full Moon
5,Phoenix,89.1,Sunny,17,"[Friday, May 1: High 95.4F / Low 67.1F - Sunny...",05:40 AM,07:11 PM,Full Moon
6,San Francisco,62.1,Partly cloudy,70,"[Friday, May 1: High 63.7F / Low 51.8F - Cloud...",06:14 AM,08:00 PM,Full Moon
7,Seattle,71.1,Sunny,39,"[Friday, May 1: High 73.8F / Low 49.3F - Sunny...",05:52 AM,08:22 PM,Full Moon
8,Austin,57.9,Light rain,90,"[Friday, May 1: High 61.3F / Low 56.1F - Heavy...",06:48 AM,08:09 PM,Full Moon
9,Denver,60.1,Partly cloudy,15,"[Friday, May 1: High 61.1F / Low 40.9F - Patch...",06:00 AM,07:54 PM,Full Moon
